# 19. Extending the pipeline: solutions

Exercises of [chapter 19](../../19_extending.md). Run the setup cell first. The pipeline is changed only inside this session, and the changes are undone; no file of the pipeline is edited.

These are the solutions; try the exercise notebook first.

In [ ]:
# Setup: run this cell first, in every notebook
import os, subprocess, sys
from pathlib import Path

# find the repository: from this folder upwards, the first one holding data_acquisition/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_acquisition").is_dir())
sys.path.insert(0, str(ROOT))                  # so that Python can import the pipeline
SAMPLE = ROOT / "output/manual/sample_root"    # the course's invented dataset
if not SAMPLE.exists():                        # generated once, with this same Python
    subprocess.run([sys.executable, str(ROOT / "docs/manual/tools/make_sample.py"), "--out", str(SAMPLE)], check=True)
DATASET = SAMPLE / "repo/doi_10.5555_training.pond"    # the sample's dataset folder
EXTRACT = ROOT / "docs/manual/data/montana_extract"    # a real piece of Rumney Ranch
os.environ["BEAVER_SITES"] = str(SAMPLE / "sites.yaml")    # use the sample's site registry, not the real one
import warnings; warnings.simplefilter("ignore")           # hide warnings; remove this line when debugging
print("repository:", ROOT); print("sample data root:", SAMPLE)

In [ ]:
import copy, datetime, shutil, tempfile, textwrap
import yaml
from data_acquisition.catalog import build
from data_acquisition.catalog import summary as summaries
from data_acquisition.steps import curate

REFERENCE = yaml.safe_load((ROOT / "docs/manual/solutions/training_pond.yaml").read_text())

**1. A toy product, end to end.** Add a product `water_depth` (a raster of depths in metres, band name `depth`) to `PRODUCTS` and `METRIC_BANDS` for this session. In a scratch copy of the sample, describe the DSM as a `water_depth` (variant `toy`) too, build and curate. What does the item get from the new entries? Then undo them.

> **From the chapter.** §2, the row on a product.
>
> **Code.** `summaries.PRODUCTS` in [`data_acquisition/catalog/summary.py`](../../../../data_acquisition/catalog/summary.py), `build.METRIC_BANDS` in [`data_acquisition/catalog/build.py`](../../../../data_acquisition/catalog/build.py), and `curate.curate_item` in [`data_acquisition/steps/curate.py`](../../../../data_acquisition/steps/curate.py).

In [ ]:
summaries.PRODUCTS.add("water_depth")
build.METRIC_BANDS["water_depth"] = "depth"
try:
    root = Path(tempfile.mkdtemp()) / "root"
    shutil.copytree(SAMPLE, root, ignore=shutil.ignore_patterns("stac"))
    s = copy.deepcopy(REFERENCE)
    s["groups"][0]["files"].append({"path": "dsm_2023-07-12.tif", "product": "water_depth", "variant": "toy",
                                    "date": datetime.date(2023, 7, 12)})
    path = Path(tempfile.mkdtemp()) / "summary.yaml"
    path.write_text(yaml.safe_dump(s, sort_keys=False))
    s = summaries.load(str(path))
    catalog, report = build.build(root, s)
    item = next(i for i in catalog.get_items(recursive=True) if i.id.endswith("water_depth_toy"))
    curate.curate_item(item, s)
    asset = item.assets["water_depth"]
    print(item.id)
    print("bands:", asset.extra_fields.get("eo:bands"), "| raster:", asset.extra_fields.get("raster:bands"))
    print("flags:", [q["flag"] for q in item.properties["beaver:quality"]])
finally:
    summaries.PRODUCTS.discard("water_depth")
    del build.METRIC_BANDS["water_depth"]
print("undone:", "water_depth" not in summaries.PRODUCTS and "water_depth" not in build.METRIC_BANDS)
# Two lines made a product: its band is named "depth" with unit m (METRIC_BANDS). No plausibility check applies:
# curate's elevation range is for dsm and dtm only, and a depth range would be a new check (chapter 15).

**2. A test for it.** Write the test that would go with the change, using pytest's `monkeypatch` fixture so that the additions are undone after the test, and run it.

> **From the chapter.** §4: the checklist. [Chapter 18](../../18_testing.md), §1 explains the tests.
>
> **Code.** `pytest`'s `monkeypatch` fixture, on `summaries.PRODUCTS` in [`data_acquisition/catalog/summary.py`](../../../../data_acquisition/catalog/summary.py) and `build.METRIC_BANDS` in [`data_acquisition/catalog/build.py`](../../../../data_acquisition/catalog/build.py); the repository's settings are in [`pytest.ini`](../../../../pytest.ini).

In [ ]:
scratch = Path(tempfile.mkdtemp())
(scratch / "test_water_depth.py").write_text(textwrap.dedent('''
    from data_acquisition.catalog import build
    from data_acquisition.catalog import summary as summaries


    def test_water_depth_is_a_metric_product(monkeypatch):
        monkeypatch.setattr(summaries, "PRODUCTS", summaries.PRODUCTS | {"water_depth"})
        monkeypatch.setitem(build.METRIC_BANDS, "water_depth", "depth")
        header = {"count": 1, "colorinterp": ["gray"]}
        assert build._bands("water_depth", None, header, None) == (["depth"], [])
        problems = summaries.check({"schema": 1, "dataset": "t", "title": "t",
                                    "authored": {"by": "me", "date": "2026-10-19"}, "source": {"kind": "folder",
                                    "provider": "p"}, "groups": [{"folder": "f", "site": "s", "files": [
                                    {"path": "d.tif", "product": "water_depth", "date": "2023-07-12"}]}]}, {})
        assert not any("product must be" in p for p in problems)
'''))
out = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-c", str(ROOT / "pytest.ini"),
                      "--rootdir", str(ROOT), str(scratch / "test_water_depth.py")],
                     cwd=ROOT, capture_output=True, text=True)
print((out.stdout + out.stderr).strip().splitlines()[-1])
# In the real change, the two additions would be edits to summary.py and build.py, and the test would need no
# monkeypatch; here it keeps the pipeline untouched.

**3. Data or code?** For each case, say whether it is data (which file) or code (which row of §2): a new Zenodo dataset; a repository serving records in DataCite XML; a site's colony boundaries; a survey whose rasters are in feet; a dataset of LiDAR point clouds; a summary value Claude got wrong.

> **From the chapter.** §2: the table of extensions, and §4: the first question of the checklist.
>
> **Code.** No code: answer from the table, with the files it names.

In [ ]:
answers = {
    "a new Zenodo dataset": "data: a summary (the Zenodo client exists)",
    "records in DataCite XML": "code: a metadata standard (readers/datacite.py + RECORDS), and a client if the "
                               "repository is new",
    "a site's colony boundaries": "data: the site registry (colonies, entered by a person), later define_aoi",
    "rasters in feet": "data: the summary's `unit` (and a note); conversion is a later step",
    "LiDAR point clouds": "code: a data type (reader, builder kind, STAC item, evidence, check); plan with the design",
    "a wrong summary value": "data: correct the summary, with its evidence, and review it again",
}
for case, answer in answers.items():
    print(f"{case:28s} {answer}")